Extracting API levels: YMLs

    - Part 1: YML files parse and check for keyword detection in general

    - Part 2: YML files parse and check for execusion environments require or optionally need api and then search for api-levels


In [1]:
# API Level Extraction from YAML Files
# YAML API - Part 1 - keyword matching
import os
import yaml
import pandas as pd
import re

# === CONFIG ===
PROJECTS_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SUMMARY_CSV = os.path.join(OUTPUT_DIR, "4.1_API_YML_Summary_1.csv")
DETAILED_CSV = os.path.join(OUTPUT_DIR, "4.1_API_YML_Details_1.csv")


# === EXTRACT ALL API LEVELS (unchanged) ===
def extract_all_api_levels(obj):
    api_levels = set()

    def recurse(o):
        if isinstance(o, dict):
            for k, v in o.items():
                key_lower = str(k).lower()

                # ✅ EXISTING: Strict match for known SDK/API-related keys
                if re.fullmatch(r'(api[-_]?level|api[-_]?versions|compile[-_]?sdk|target[-_]?sdk)', key_lower):
                    if isinstance(v, list):
                        for val in v:
                            if isinstance(val, str):
                                split_vals = re.split(r'[,\s]+', val)
                                for item in split_vals:
                                    if item.strip().isdigit():
                                        api_levels.add(item.strip())
                            elif isinstance(val, int):
                                api_levels.add(str(val))
                    elif isinstance(v, str):
                        split_vals = re.split(r'[,\s]+', v)
                        for item in split_vals:
                            if item.strip().isdigit():
                                api_levels.add(item.strip())
                    elif isinstance(v, int):
                        api_levels.add(str(v))

                # ✅ NEW: Relaxed key match for variables like ANDROID_EMULATOR_API_LEVEL
                elif re.fullmatch(r'(compile[_-]?sdk|target[_-]?sdk|api[_-]?level)', key_lower):
                    if isinstance(v, (int, str)) and str(v).isdigit():
                        api = int(v)
                        if 1 <= api <= 35:  # allow current APIs
                            api_levels.add(str(api))


                # Handle lists
                if isinstance(v, list):
                    for val in v:
                        if isinstance(val, dict):
                            recurse(val)
                        elif isinstance(val, str):
                            matches = re.findall(
                                r'(?:api[-_]?level\s*[:=]?\s*|compile[-_]?sdk\s*[:=]?\s*|target[-_]?sdk\s*[:=]?\s*|platforms;android[-_]?|android-)(\d{2,3})',
                                val,
                                flags=re.IGNORECASE
                            )
                            for match in matches:
                                api_levels.add(match)

                elif isinstance(v, dict):
                    recurse(v)

                elif isinstance(v, str):
                    recurse(v)

        elif isinstance(o, list):
            for item in o:
                recurse(item)

        elif isinstance(o, str):
            if any(kw in o.lower() for kw in ['--flavor', 'flutter build', 'gradlew', 'apk', 'assemble']):
                return

            matches = re.findall(
                r'(?:api[-_]?level\s*[:=]?\s*|compile[-_]?sdk\s*[:=]?\s*|target[-_]?sdk\s*[:=]?\s*)["\']?(\d{2,3})["\']?',
                o,
                flags=re.IGNORECASE
            )
            for match in matches:
                api_levels.add(match)

            sdkmanager_matches = re.findall(
                r'platforms;android[-_]?(\d{2,3})',
                o,
                flags=re.IGNORECASE
            )
            for match in sdkmanager_matches:
                api_levels.add(match)

    recurse(obj)
    return api_levels


# === PARSE YAML FILE (unchanged) ===
def parse_yaml_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw = f.read().replace('\t', ' ')
            doc = yaml.safe_load(raw) or {}

        jobs = (doc.get('jobs') or {})
        instr = None

        # 1) Prefer the canonical job name if present
        if 'instrumentation-tests' in jobs:
            instr = jobs['instrumentation-tests']

        # 2) Fallback: find a job that looks like an emulator/instrumentation run
        if instr is None:
            def looks_like_instr(job_dict):
                if not isinstance(job_dict, dict):
                    return False
                # check for matrix.api-level or steps.*.with.api-level
                matrix = job_dict.get('strategy', {}).get('matrix', {})
                if any(k in matrix for k in ('api-level', 'api_level', 'apiLevels', 'apiLevels'.lower())):
                    return True
                # scan steps for reactivecircus or api-level under with:
                for step in job_dict.get('steps', []) or []:
                    if isinstance(step, dict):
                        if isinstance(step.get('uses'), str) and 'reactivecircus/android-emulator-runner' in step['uses'].lower():
                            return True
                        with_block = step.get('with') or {}
                        if any(k.replace('_','-').lower() == 'api-level' for k in with_block.keys()):
                            return True
                return False

            for job_name, job_def in jobs.items():
                if looks_like_instr(job_def):
                    instr = job_def
                    break

        # Nothing instrumentation-like found
        if instr is None:
            return {'api_levels': set(), 'error': False}

        # Only extract API levels from the instrumentation job subtree
        all_api_levels = extract_all_api_levels(instr)
        return {'api_levels': all_api_levels, 'error': False}

    except Exception:
        return {'api_levels': set(), 'error': True}



# === SCAN PROJECTS ===
project_results = {}
detailed_rows = {}

for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            file_path = os.path.join(root, file)
            filename = os.path.basename(file_path)

            # ✅ Extract full_name before first "__"
            full_name = filename.split("__")[0].lower()
            result = parse_yaml_file(file_path)

            if full_name not in project_results:
                project_results[full_name] = {
                    'api_levels': set(),
                    'errors': 0,
                    'yml_count': 0
                }
                detailed_rows[full_name] = []

            project_results[full_name]['api_levels'].update(result['api_levels'])
            project_results[full_name]['yml_count'] += 1
            if result['error']:
                project_results[full_name]['errors'] += 1

# === BUILD DETAILED ROWS ===
final_detailed_rows = []
for full_name, data in project_results.items():
    for api in data['api_levels']:
        final_detailed_rows.append({
            'filename': '',  # placeholder for below loop
            'full_name': full_name,
            'api_level': api,
            'source': 'detected',
            'yml_count': data['yml_count']
        })

# === Rebuild detailed rows with filename ===
# Note: Since each full_name may come from multiple files,
# we'll loop again to map filename properly.

detailed_output = []
for root, _, files in os.walk(PROJECTS_DIR):
    for file in files:
        if file.endswith(('.yml', '.yaml')):
            filename = os.path.basename(file)
            full_name = filename.split("__")[0].lower()

            result = parse_yaml_file(os.path.join(root, file))
            for api in result['api_levels']:
                detailed_output.append({
                    'filename': filename,
                    'full_name': full_name,
                    'api_level': api,
                    'source': 'yml',
                })


# === EXPORT ===
df_detailed = pd.DataFrame(detailed_output)
df_detailed.to_csv(DETAILED_CSV, index=False)

summary_rows = []
for full_name, result in project_results.items():
    summary_rows.append({
        'full_name': full_name,
        'distinct_api_levels': len(result['api_levels']),
        'yaml_errors': result['errors'],
        'yaml_count': result['yml_count'],
    })

pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)

print(f"\n✅ Summary CSV saved to: {SUMMARY_CSV}")
print(f"✅ Detailed CSV saved to: {DETAILED_CSV}")



✅ Summary CSV saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_API_YML_Summary_1.csv
✅ Detailed CSV saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_API_YML_Details_1.csv



##### Part 2 - execution environment setup detection to api
 Device Setup's check analysis for api level detectoin

In [2]:
# -*- coding: utf-8 -*-
# Part 2 - execution environment setup detection to api (final)
import os, re, yaml, pandas as pd
from typing import Any, Dict, List, Set, Tuple

# === CONFIG ===
PROJECTS_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_DIR    = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
os.makedirs(OUTPUT_DIR, exist_ok=True)

General_CSV   = os.path.join(OUTPUT_DIR, "4.1_API_YML_Conditioned.csv")
CI_Format_CSV = os.path.join(OUTPUT_DIR, "4.1_API_YML_Details_2.csv")

API_MAX = 999  # clamp later if you want (e.g., <= 34)

# --- Device-setup patterns ---
PATTERN_EMULATOR = [
    r'(?m)\bavdmanager\b.*\b(create|delete)\b',
    r'(?m)\bemulator\b.*(-avd|@)\S+',
    r'(?m)^\s*circle-android\s+wait-for-boot\b',   # CircleCI helper
    r'(?m)\bandroid-wait-for-emulator\b',
    r'(?m)\breactivecircus/android-emulator-runner\b',
    r'(?m)\bsdkmanager\b.*\b(system-images;android-\S+|emulator|platforms;android-\S+)\b',
]
PATTERN_GMD = [
    r'(?m)\bmanageddevices?\b',
    r'(?m)\bmanagedvirtualdevice\b',
    r'(?m)^\s*arguments\s*:\s*[:\w-]*manageddevice\w*androidtest\b',
    r'(?m)^\s*tasks?\s*:\s*[:\w-]*manageddevice\w*androidtest\b',
]
PATTERN_LABS = [
    r'(?m)\bgcloud\b.*\bfirebase\s+test\s+android\s+run\b',
    r'(?m)\bappcenter\s+test\s+run\s+android\b',
    r'(?m)\bsaucectl(\s+run)?\b',
    r'(?m)\b(browserstack|bstack)\b',
]

DEVICE_SETUP_PATTERNS = {
    "Emulator": [re.compile(p, re.I) for p in PATTERN_EMULATOR],
    "GMD":      [re.compile(p, re.I) for p in PATTERN_GMD],
    "CloudLab": [re.compile(p, re.I) for p in PATTERN_LABS],
}

API_REQUIRED = {"Emulator", "GMD"}   # where api-level is required
API_OPTIONAL = {"CloudLab"}          # optional (real devices omitted here)

# tokens: $VAR, ${VAR}, or $28
VAR_TOKEN_RE = re.compile(r'\$(\{?[A-Za-z_][A-Za-z0-9_]*\}?|\d+)')

# literal api digit detection (after any interpolation)
LIT_API_RE_LIST = [
    re.compile(r'(?:platforms;android-|system-images;android-)(\d{2,3})', re.I),
    re.compile(r'\bsys-img-[^\s]*?android-?(\d{2,3})\b', re.I),  # older sdkmanager/android tool syntax
    re.compile(r'\bapi[-_ ]?level\s*[:=]\s*["\']?(\d{2,3})["\']?', re.I),
    re.compile(r'\bandroid-?(\d{2,3})\b', re.I),                 # matches android21 or android-21
    re.compile(r'\bapiLevel\s*[:=]\s*["\']?(\d{2,3})["\']?', re.I),
    # Pull API from AVD names used with -avd (circleci-android21, Pixel_API_30, etc.)
    re.compile(r'\bemulator\b[^\n]*?-avd\s+\S*?(?:android-?|api[_-]?)(\d{2,3})\b', re.I),
]

def iter_strings(o: Any):
    """Yield all strings from YAML, including dict KEYS and VALUES."""
    if isinstance(o, dict):
        for k, v in o.items():
            if isinstance(k, str):
                yield k
            yield from iter_strings(v)
    elif isinstance(o, list):
        for x in o:
            yield from iter_strings(x)
    elif isinstance(o, str):
        yield o

def collect_env_vars(root: Any) -> Dict[str, Set[str]]:
    """
    Collect env vars from Travis-style lists and GitHub Actions mapping-style env blocks.
    Robust to odd quoting, no shlex.
    Returns: {VAR_NAME: {value1, value2, ...}}
    """
    env_map: Dict[str, Set[str]] = {}

    def add_kv(key: str, val: Any):
        if isinstance(val, (int, float, bool)) or val is None:
            val = "" if val is None else str(val)
        elif not isinstance(val, str):
            val = str(val)
        env_map.setdefault(key, set()).add(val)

    def add_kv_string(s: str):
        # Extract KEY=VALUE tokens (VALUE may be "..." or '...' or bare)
        for m in re.finditer(
            r'([A-Za-z_][A-Za-z0-9_]*)='
            r'(?:'
            r'"([^"]*)"'
            r"|\'([^\']*)\'"
            r"|([^\s]+)"
            r')',
            s
        ):
            key = m.group(1)
            val = next(v for v in m.groups()[1:] if v is not None)
            add_kv(key, val)

    def walk(o: Any):
        if isinstance(o, dict):
            for k, v in o.items():
                if str(k).lower() == 'env':
                    if isinstance(v, dict):
                        handled_nested = False
                        for sub in ('global', 'matrix'):
                            if sub in v:
                                handled_nested = True
                                subval = v[sub]
                                if isinstance(subval, dict):
                                    for kk, vv in subval.items():
                                        add_kv(kk, vv)
                                elif isinstance(subval, list):
                                    for item in subval:
                                        if isinstance(item, dict):
                                            for kk, vv in item.items():
                                                add_kv(kk, vv)
                                        elif isinstance(item, str):
                                            add_kv_string(item)
                                elif isinstance(subval, str):
                                    add_kv_string(subval)
                        if not handled_nested:
                            for kk, vv in v.items():
                                add_kv(kk, vv)
                    elif isinstance(v, list):
                        for item in v:
                            if isinstance(item, dict):
                                for kk, vv in item.items():
                                    add_kv(kk, vv)
                            elif isinstance(item, str):
                                add_kv_string(item)
                    elif isinstance(v, str):
                        add_kv_string(v)
                # Recurse
                walk(v)
        elif isinstance(o, list):
            for x in o:
                walk(x)

    walk(root)
    return env_map  # var -> set(values)

def find_device_setup_lines(all_strings: List[str]) -> Dict[str, List[str]]:
    hits: Dict[str, List[str]] = {"Emulator":[], "GMD":[], "CloudLab":[]}
    for s in all_strings:
        for dtype, regs in DEVICE_SETUP_PATTERNS.items():
            if any(rx.search(s) for rx in regs):
                hits[dtype].append(s)
    return hits

def interpolate_vars_in_text(s: str, single_env_map: Dict[str, str]) -> str:
    def repl(m):
        token = m.group(1)
        if token.startswith('{') and token.endswith('}'):
            token = token[1:-1]
        if token.isdigit():  # $28 -> 28
            return token
        return single_env_map.get(token, m.group(0))  # keep as-is if unknown
    return VAR_TOKEN_RE.sub(repl, s)

def extract_literals(text: str) -> Set[str]:
    out: Set[str] = set()
    for rx in LIT_API_RE_LIST:
        for m in rx.findall(text):
            if isinstance(m, tuple):
                # safety if a regex had multiple groups accidentally
                m = next((g for g in m if g), "")
            if m and str(m).isdigit():
                n = int(m)
                if 1 <= n <= API_MAX:
                    out.add(str(n))
    return out

def analyze_file(path: str) -> Tuple[List[Dict[str, str]], List[Dict[str, str]]]:
    """
    Returns:
      primary_rows: one summary row per file
      secondary_rows: many detail rows (env-resolved, literal, variable_api)
    """
    try:
        raw = open(path, 'r', encoding='utf-8', errors='ignore').read().replace('\t',' ')
        data = yaml.safe_load(raw)
        if not data:
            return [], []
    except Exception:
        return [], []

    strings = list(iter_strings(data))
    env_vars = collect_env_vars(data)  # var -> set(values)
    ds_hits = find_device_setup_lines(strings)
    present_types = {t for t, lines in ds_hits.items() if lines}

    # containers
    unresolved_vars: Set[str] = set()
    accepted_api: Set[str] = set()
    literal_api: Set[str] = set()

    # single map for quick interpolation (first value per var)
    single_env = {k: next(iter(v)) for k, v in env_vars.items() if v}

    secondary_rows: List[Dict[str, str]] = []
    filename = os.path.basename(path)

    # 1) Extract from the lines that flagged device setup
    for dtype, lines in ds_hits.items():
        for s in lines:
            # literals directly in the line (no var)
            lit_now = extract_literals(interpolate_vars_in_text(s, {}))  # also catches $28→28
            if lit_now:
                literal_api |= lit_now
                for a in sorted(lit_now):
                    secondary_rows.append({
                        "filename": filename,
                        "device_type": dtype,
                        "api_required": str(dtype in API_REQUIRED),
                        "source": "literal",
                        "key": "",
                        "api_level": a,
                        "example_line": s[:500],
                    })

            # variable refs in the line
            var_refs: List[str] = []
            for m in VAR_TOKEN_RE.finditer(s):
                token = m.group(1)
                if token.startswith('{') and token.endswith('}'):
                    token = token[1:-1]
                if not token.isdigit():  # exclude numeric tokens (already counted as literals above)
                    var_refs.append(token)

            for var in var_refs:
                if var in env_vars:
                    for val in env_vars[var]:
                        # try concrete substitution
                        text = interpolate_vars_in_text(s, {var: val})
                        found = extract_literals(text)
                        if found:
                            for a in sorted(found):
                                accepted_api.add(a)
                                secondary_rows.append({
                                    "filename": filename,
                                    "device_type": dtype,
                                    "api_required": str(dtype in API_REQUIRED),
                                    "source": "env_resolved",
                                    "key": var,
                                    "api_level": a,
                                    "example_line": s[:500],
                                })
                        else:
                            # fallback: accept numeric env value itself
                            if val.isdigit() and 1 <= int(val) <= API_MAX:
                                accepted_api.add(val)
                                secondary_rows.append({
                                    "filename": filename,
                                    "device_type": dtype,
                                    "api_required": str(dtype in API_REQUIRED),
                                    "source": "env_resolved",
                                    "key": var,
                                    "api_level": val,
                                    "example_line": s[:500],
                                })
                else:
                    unresolved_vars.add(var)
                    secondary_rows.append({
                        "filename": filename,
                        "device_type": dtype,
                        "api_required": str(dtype in API_REQUIRED),
                        "source": "variable_api",
                        "key": var,
                        "api_level": "",
                        "example_line": s[:500],
                    })

    # 2) Context-wide scan for API literals (to catch api-level lines separate from 'uses')
    if present_types:
        joined = "\n".join(strings)
        ctxt_api = extract_literals(joined)
        for a in sorted(ctxt_api - literal_api - accepted_api):
            # only log as context if emulator/GMD present (to keep signal tight)
            secondary_rows.append({
                "filename": filename,
                "device_type": ",".join(sorted(present_types)),
                "api_required": str(bool(present_types & API_REQUIRED)),
                "source": "context",
                "key": "",
                "api_level": a,
                "example_line": "",
            })
        literal_api |= (ctxt_api - accepted_api)

    api_required_here = bool(present_types & API_REQUIRED)

    primary_rows = [{
        "filename": filename,
        "device_types_detected": ", ".join(sorted(present_types)) if present_types else "",
        "api_required_for_present_types": str(api_required_here),
        "api_levels_from_literals": ", ".join(sorted(literal_api)) if literal_api else "",
        "api_levels_from_env_resolved": ", ".join(sorted(accepted_api)) if accepted_api else "",
        "unresolved_variable_api": ", ".join(f"variable_api:{v}" for v in sorted(unresolved_vars)) if unresolved_vars else "",
        "example_device_setup_line": (ds_hits[next(iter(present_types))][0][:500] if present_types else "")
    }]

    return primary_rows, secondary_rows

# === RUN over directory ===
primary_all: List[Dict[str, str]] = []
secondary_all: List[Dict[str, str]] = []

for root, _, files in os.walk(PROJECTS_DIR):
    for fname in files:
        if not fname.lower().endswith(('.yml', '.yaml')):
            continue
        fpath = os.path.join(root, fname)
        prim, sec = analyze_file(fpath)
        primary_all.extend(prim)
        secondary_all.extend(sec)

# Save primary (summary)
pd.DataFrame(primary_all, columns=[
    "filename",
    "device_types_detected",
    "api_required_for_present_types",
    "api_levels_from_literals",
    "api_levels_from_env_resolved",
    "unresolved_variable_api",
    "example_device_setup_line",
]).to_csv(General_CSV, index=False)

# Save secondary (merged details)
pd.DataFrame(secondary_all, columns=[
    "filename",
    "device_type",
    "api_required",
    "source",       # literal | env_resolved | variable_api | context
    "key",          # env var name (if any)
    "api_level",    # api level (if resolved)
    "example_line",
]).to_csv(CI_Format_CSV, index=False)

print(f"Saved primary summary -> {General_CSV} (rows={len(primary_all)})")
print(f"Saved secondary details -> {CI_Format_CSV} (rows={len(secondary_all)})")


Saved primary summary -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_API_YML_Conditioned.csv (rows=12627)
Saved secondary details -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_API_YML_Details_2.csv (rows=974)


In [3]:
# correcting columns in the output file

# -*- coding: utf-8 -*-
import os
import pandas as pd

BASE_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
IN_FILE  = os.path.join(BASE_DIR, "4.1_API_YML_Details_2.csv")
OUT_FILE = os.path.join(BASE_DIR, "4.1_API_YML_Details_2.csv")

def pick_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of the columns {candidates} found in: {list(df.columns)}")

def derive_full_name(filename: str) -> str:
    base = os.path.basename(str(filename))
    if "__" in base:
        full = base.split("__", 1)[0]
    else:
        full = os.path.splitext(base)[0]
    return full.lower()

def main():
    df = pd.read_csv(IN_FILE)

    # Flexible column detection
    filename_col = pick_column(df, ["filename", "file_name", "file"])
    apilevel_col = pick_column(df, ["api_level", "apiLevel", "api level"])

    out = df[[filename_col, apilevel_col]].copy()
    out.columns = ["filename", "api_level"]  # normalize

    # 1) Drop NaN api_level rows BEFORE converting to string
    out = out[out["api_level"].notna()]

    # 2) Clean strings and remove empty or "nan" leftovers
    out["filename"]  = out["filename"].astype(str).str.strip()
    out["api_level"] = out["api_level"].astype(str).str.strip()
    mask_nonempty = out["api_level"].ne("") & out["api_level"].str.lower().ne("nan")
    out = out[mask_nonempty]

    # 3) Derive full_name (lowercase before "__")
    out["full_name"] = out["filename"].apply(derive_full_name)

    # 4) Add constant source column
    out["source"] = "yml"

    # 5) Reorder columns
    out = out[["filename", "full_name", "api_level", "source"]]

    # 6) Deduplicate records
    out = out.drop_duplicates(subset=["filename", "full_name", "api_level"])

    out.to_csv(OUT_FILE, index=False)
    print(f"Saved: {OUT_FILE} (rows={len(out)})")

if __name__ == "__main__":
    main()


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_API_YML_Details_2.csv (rows=520)


In [4]:
# Merging YML API-levels from Part 1 and Part 2 into a single file
# -*- coding: utf-8 -*-
import os
import pandas as pd
from glob import glob
from typing import Optional

BASE_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
IN_FILES = [
    os.path.join(BASE_DIR, "4.1_API_YML_Details_1.csv"),
    os.path.join(BASE_DIR, "4.1_API_YML_Details_2.csv"),
]
OUT_FILE = os.path.join(BASE_DIR, "4.1_API_YML_Details.csv")

# Canonical column names we want in the final file
CANONICAL = ["filename", "full_name", "api_level", "source"]

# Flexible aliases (case-insensitive)
ALIASES = {
    "filename":  ["filename", "file_name", "file"],
    "full_name": ["full_name", "name"],
    "api_level": ["api_level", "apilevel", "api level", "api"],
    "source":    ["source"],
}

def pick_first_existing(df: pd.DataFrame, names: list[str]) -> Optional[str]:
    """Return the actual column name in df that matches any of `names` (case-insensitive)."""
    lower_map = {c.lower(): c for c in df.columns}
    for n in names:
        cn = lower_map.get(n.lower())
        if cn is not None:
            return cn
    return None

def derive_full_name(filename: str) -> str:
    base = os.path.basename(str(filename))
    if "__" in base:
        full = base.split("__", 1)[0]
    else:
        full = os.path.splitext(base)[0]
    return full.strip().lower()

def normalize(df: pd.DataFrame) -> pd.DataFrame:
    # Map to canonical columns (missing -> empty)
    cols: dict[str, pd.Series] = {}
    for canon in CANONICAL:
        col = pick_first_existing(df, ALIASES[canon])
        if col is not None:
            cols[canon] = df[col]
        else:
            cols[canon] = pd.Series([""] * len(df), index=df.index)

    out = pd.DataFrame(cols)

    # Clean strings
    for c in CANONICAL:
        out[c] = out[c].astype(str).str.strip()

    # Full name: lowercase for ALL rows
    out["full_name"] = out["full_name"].str.lower()

    # Derive full_name where blank
    mask_blank_full = out["full_name"].eq("") | out["full_name"].str.lower().eq("nan")
    if mask_blank_full.any():
        out.loc[mask_blank_full, "full_name"] = (
            out.loc[mask_blank_full, "filename"].apply(derive_full_name)
        )

    # Extract just the numeric API level (2–3 digits), drop rows where missing
    api_digits = out["api_level"].str.extract(r"(\d{2,3})", expand=False)
    out["api_level"] = api_digits.fillna("").astype(str).str.strip()
    out = out[out["api_level"].ne("")]

    # Ensure source present; default to 'yml' if blank
    blank_src = out["source"].eq("") | out["source"].str.lower().eq("nan")
    out.loc[blank_src, "source"] = "yml"

    # Final column order
    out = out[CANONICAL]
    return out

def main():
    # Collect inputs: explicit list or any matching pattern
    files = [f for f in IN_FILES if os.path.isfile(f)]
    if not files:
        files = [f for f in glob(os.path.join(BASE_DIR, "4.1_API_YML_Details_*.csv")) if os.path.isfile(f)]
    if not files:
        raise FileNotFoundError("No input files found to merge.")

    parts = []
    for f in files:
        df = pd.read_csv(f, dtype=str).fillna("")  # keep strings, avoid literal "nan"
        parts.append(normalize(df))

    merged = pd.concat(parts, ignore_index=True)

    # Deduplicate across canonical cols
    before = len(merged)
    merged = merged.drop_duplicates(subset=CANONICAL)
    after = len(merged)

    merged.to_csv(OUT_FILE, index=False)
    print(f"Merged {len(files)} file(s). Saved: {OUT_FILE} (rows={after}, dropped_dupes={before - after})")

if __name__ == "__main__":
    main()


Merged 2 file(s). Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_API_YML_Details.csv (rows=969, dropped_dupes=5)


In [5]:
import os
import re
import pandas as pd

# === CONFIG ===
BUILD_DIR = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"

os.makedirs(OUTPUT_DIR, exist_ok=True)

SUMMARY_CSV  = os.path.join(OUTPUT_DIR, "4.1_API_Gradle_Summary.csv")
DETAILED_CSV = os.path.join(OUTPUT_DIR, "4.1_API_Gradle_Details.csv")

# === HELPERS ===
def is_build_gradle_file(fname: str) -> bool:
    """
    Accept ONLY Gradle build files, including numbered variants saved with '++':
      - build.gradle / build.gradle.kts
      - build__<n>.gradle / build__<n>.gradle.kts
      - owner.repo__Type++build__<n>.gradle[.kts]
    """
    base = os.path.basename(fname)

    # Saved form: owner.repo__Type++<tail>
    if "++" in base:
        tail = base.split("++", 1)[1]
        # normalize '__<digits>' immediately before extension
        tail = re.sub(r"__\d+(?=\.gradle(?:\.kts)?$)", "", tail, flags=re.IGNORECASE).lower()
        return tail in ("build.gradle", "build.gradle.kts")

    # Plain filename (rare in All_Config_Files, but supported)
    return bool(re.match(r"(?i)^build(?:__\d+)?\.gradle(?:\.kts)?$", base))

def extract_full_name_from_saved(fname: str, fpath: str) -> str:
    """
    Extract owner.repo from saved filenames like owner.repo__Type++<tail>.
    Fallback to parent directory name if pattern not present.
    """
    base = os.path.basename(fname)
    if "__" in base:
        return base.split("__", 1)[0].lower()
    return os.path.basename(os.path.dirname(fpath)).lower()

def strip_comments_gradle(text: str) -> str:
    """
    Remove block comments /* ... */ and line comments // ... .
    Keeps http(s):// by not stripping '//' if preceded by ':'.
    """
    if not text:
        return ""
    s = re.sub(r"/\*.*?\*/", "", text, flags=re.DOTALL)
    s = re.sub(r"(?<!:)//.*?$", "", s, flags=re.MULTILINE)
    return s

# === EXTRACT ONLY INSTRUMENTATION-RELATED API LEVELS ===
def extract_api_levels_from_gradle(content: str):
    api_levels = set()
    content = content.replace('\t', '    ')
    # Match managedDevices apiLevel only
    pattern = re.compile(r'\bapiLevel\b\s*(?:[=:])?\s*["\']?(\d{2,3})["\']?', flags=re.IGNORECASE)
    for m in pattern.finditer(content):
        v = m.group(1)
        if v.isdigit():
            api_levels.add(v)
    return api_levels

# === SCAN GRADLE FILES ===
project_results = {}
detailed_output = []

for root, _, files in os.walk(BUILD_DIR):
    for file in files:
        if not is_build_gradle_file(file):
            continue

        fpath = os.path.join(root, file)
        filename = os.path.basename(fpath)
        full_name = extract_full_name_from_saved(file, fpath)

        try:
            with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
                raw = f.read()
            content = strip_comments_gradle(raw)  # avoid picking apiLevel from comments
            api_levels = extract_api_levels_from_gradle(content)
            error = False
        except Exception:
            api_levels = set()
            error = True

        if full_name not in project_results:
            project_results[full_name] = {
                'api_levels': set(),
                'errors': 0,
                'build_file_count': 0
            }

        project_results[full_name]['api_levels'].update(api_levels)
        project_results[full_name]['build_file_count'] += 1
        if error:
            project_results[full_name]['errors'] += 1

        for api in api_levels:
            detailed_output.append({
                'filename': filename,
                'full_name': full_name,
                'api_level': api,
                'source': 'gradle',
            })

# === EXPORT TO CSV ===
pd.DataFrame(detailed_output).to_csv(DETAILED_CSV, index=False)

summary_rows = [
    {
        'full_name': fn,
        'distinct_api_levels': len(res['api_levels']),
        'build_file_errors': res['errors'],
        'build_file_count': res['build_file_count'],
    }
    for fn, res in project_results.items()
]
pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)

print(f"\n✅ Instrumentation API-level summary saved to: {SUMMARY_CSV}")
print(f"✅ Detailed API-level results saved to: {DETAILED_CSV}")



✅ Instrumentation API-level summary saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_API_Gradle_Summary.csv
✅ Detailed API-level results saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_API_Gradle_Details.csv


In [6]:
# Merge YML and Gradle API Level Data (clean + dedupe)
import os
import re
import pandas as pd

FOLDER = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"

yml_details_path    = os.path.join(FOLDER, "4.1_API_YML_Details.csv")
gradle_details_path = os.path.join(FOLDER, "4.1_API_Gradle_Details.csv")
merged_details_path = os.path.join(FOLDER, "4.1_API_Merged_Details.csv")

CANONICAL = ["filename", "full_name", "api_level", "source"]
ALIASES = {
    "filename":  ["filename", "file_name", "file"],
    "full_name": ["full_name", "name"],
    "api_level": ["api_level", "apilevel", "api level", "api"],
    "source":    ["source"],
}

def pick_col(df: pd.DataFrame, names: list[str]) -> str | None:
    lower_map = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in lower_map:
            return lower_map[n.lower()]
    return None

def normalize(df: pd.DataFrame, default_source: str) -> pd.DataFrame:
    # map to canonical columns (missing -> "")
    cols = {}
    for canon in CANONICAL:
        col = pick_col(df, ALIASES[canon])
        cols[canon] = (df[col] if col is not None else pd.Series([""] * len(df), index=df.index))
    out = pd.DataFrame(cols)

    # trim strings
    for c in CANONICAL:
        out[c] = out[c].astype(str).str.strip()

    # lowercase full_name
    out["full_name"] = out["full_name"].str.lower()

    # extract 2–3 digit numeric API level
    api = out["api_level"].str.extract(r"(\d{2,3})", expand=False).fillna("")
    out["api_level"] = api

    # drop rows without api_level
    out = out[out["api_level"].ne("")].copy()

    # default source if blank
    blank_src = out["source"].eq("") | out["source"].str.lower().eq("nan")
    out.loc[blank_src, "source"] = default_source

    return out[CANONICAL]

parts = []

if os.path.isfile(yml_details_path):
    df_yml = pd.read_csv(yml_details_path, dtype=str).fillna("")
    parts.append(normalize(df_yml, default_source="yml"))
else:
    print(f"[WARN] Missing: {os.path.basename(yml_details_path)}")

if os.path.isfile(gradle_details_path):
    df_gradle = pd.read_csv(gradle_details_path, dtype=str).fillna("")
    parts.append(normalize(df_gradle, default_source="gradle"))
else:
    print(f"[WARN] Missing: {os.path.basename(gradle_details_path)}")

if not parts:
    raise FileNotFoundError("No input detail files found to merge.")

merged = pd.concat(parts, ignore_index=True)

# dedupe across canonical columns
before = len(merged)
merged = merged.drop_duplicates(subset=CANONICAL).reset_index(drop=True)
after = len(merged)

# optional: sort for readability
merged["api_level"] = merged["api_level"].astype(int)
merged = merged.sort_values(["full_name", "api_level", "source", "filename"]).reset_index(drop=True)

merged.to_csv(merged_details_path, index=False)
print(f"✅ Merged details saved to: {merged_details_path} (rows={after}, dropped_dupes={before-after})")


✅ Merged details saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_API_Merged_Details.csv (rows=1057, dropped_dupes=0)


Create the main Repo list of all the detected api levels

In [7]:
# -*- coding: utf-8 -*-
import os
import re
import pandas as pd

# === Paths ===
FOLDER = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
api_input_path = os.path.join(FOLDER, "4.1_API_Merged_Details.csv")
output_path    = os.path.join(FOLDER, "4.1_ALL_API_Levels.csv")
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# === Load API data ===
df_api = pd.read_csv(api_input_path, dtype=str).fillna("")
if "full_name" not in df_api.columns or "api_level" not in df_api.columns:
    raise ValueError("Input must contain 'full_name' and 'api_level' columns.")

# Normalize
df_api["full_name"] = df_api["full_name"].str.strip().str.lower()

# Extract numeric API level if present (handles 'android-34', ' 33 ', etc.)
api_digits = df_api["api_level"].astype(str).str.strip().str.extract(r"(\d{2,3})", expand=False)
df_api["api_level"] = api_digits.fillna("")

# Keep only valid rows
df_api = df_api[(df_api["full_name"] != "") & (df_api["api_level"] != "")].copy()

# Build ordered list of levels (as integers), then back to strings for columns
levels_int = sorted({int(x) for x in df_api["api_level"].unique()})
levels_str = [str(x) for x in levels_int]

# Pivot: count occurrences per (repo x API level)
api_counts = (
    df_api.groupby(["full_name", "api_level"])
          .size()
          .unstack(fill_value=0)
          .reindex(columns=levels_str, fill_value=0)
)

# Ensure integer dtype
api_counts = api_counts.astype(int)

# Prefix columns with api_
api_cols = [f"api_{col}" for col in api_counts.columns]
api_counts.columns = api_cols

# Add totals (sum of counts) and distinct (how many API columns have > 0)
api_counts["api_level_total"] = api_counts[api_cols].sum(axis=1).astype(int)
#api_counts["api_level_distinct"] = (api_counts[api_cols] > 0).sum(axis=1).astype(int)

# Optional: min/max API level covered per repo (empty -> NA)
def _min_api(row):
    present = [lvl for lvl, col in zip(levels_int, api_cols) if row[col] > 0]
    return min(present) if present else pd.NA

def _max_api(row):
    present = [lvl for lvl, col in zip(levels_int, api_cols) if row[col] > 0]
    return max(present) if present else pd.NA

api_counts["min_api_level"] = api_counts.apply(_min_api, axis=1)
api_counts["max_api_level"] = api_counts.apply(_max_api, axis=1)

# Save
summary_df = api_counts.reset_index()  # brings back full_name as a column
summary_df.to_csv(output_path, index=False)
print(f"✅ API summary saved to: {output_path}")


✅ API summary saved to: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_ALL_API_Levels.csv


In [8]:
# append the API levels to the main Repo list (no min/max; blanks allowed)
# -*- coding: utf-8 -*-
import os
import pandas as pd

BASE_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10"
MAIN_PATH = os.path.join(BASE_DIR, "3.2_Total_Repo.csv")
API_PATH  = os.path.join(BASE_DIR, "4.1_ALL_API_Levels.csv")
OUT_PATH  = os.path.join(BASE_DIR, "4.1_Total_Repo_Dataset.csv")

def truthy(val) -> bool:
    if isinstance(val, bool):
        return val
    return str(val).strip().lower() in {"true", "1", "yes", "y", "t"}

def pick_col(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for c in candidates:
        name = lower_map.get(c.lower())
        if name is not None:
            return name
    return None

# ---- Load ----
df_main = pd.read_csv(MAIN_PATH, dtype=str).fillna("")
df_api  = pd.read_csv(API_PATH,  dtype=str).fillna("")

# ---- Normalize key for matching ----
for d in (df_main, df_api):
    if "full_name" not in d.columns:
        raise ValueError("Both files must contain a 'full_name' column.")
    d["full_name"] = d["full_name"].astype(str).str.strip().str.lower()

# ---- Flags in main ----
instru_col    = pick_col(df_main, ["instru_test", "has_androidtest", "android_test",
                                   "androidtest_exists", "has_instrumentation_tests"])
instru_ci_col = pick_col(df_main, ["instru_t_ci_signal" ])
if instru_col is None and instru_ci_col is None:
    raise ValueError("Could not find any of the instrumentation flags in 3.2_Total_Repo.")

instr_bool = df_main[instru_col].map(truthy) if instru_col else False
ci_bool    = df_main[instru_ci_col].map(truthy) if instru_ci_col else False
mask_use_api = instr_bool | ci_bool   # True -> keep API columns; False -> blank them

# ---- Pick API columns (NO min/max; allow only counts) ----
api_cols_all = [c for c in df_api.columns if c.lower() != "full_name"]
# keep all api_* columns; these include per-level counts and api_level_total
api_cols = sorted({c for c in api_cols_all if c.lower().startswith("api_")})
if not api_cols:
    raise ValueError(f"No API columns found in {os.path.basename(API_PATH)}. Expected columns like 'api_23', 'api_level_total', etc.")

# ---- Aggregate per repo (sum counts) ----
df_api_num = df_api[["full_name"] + api_cols].copy()
# coerce to numeric before summing
for c in api_cols:
    df_api_num[c] = pd.to_numeric(df_api_num[c], errors="coerce")
agg_num = df_api_num.groupby("full_name", as_index=False)[api_cols].sum(min_count=1)

# Keep nullable ints so missing values remain blank on CSV
for c in api_cols:
    agg_num[c] = pd.to_numeric(agg_num[c], errors="coerce").astype("Int64")

# ---- Merge into main ----
df_merged = df_main.merge(agg_num, on="full_name", how="left")

# ---- Apply mask: for repos without instru/ci, blank out API columns (set NA) ----
present_num_cols = [c for c in api_cols if c in df_merged.columns]
if present_num_cols:
    # Ensure nullable integer dtype, then assign pd.NA (renders blank in CSV)
    df_merged[present_num_cols] = df_merged[present_num_cols].apply(
        lambda s: pd.to_numeric(s, errors="coerce").astype("Int64")
    )
    df_merged.loc[~mask_use_api, present_num_cols] = pd.NA

# ---- Save ----
df_merged.to_csv(OUT_PATH, index=False)
print(f"✅ Saved: {OUT_PATH}")
print(f"Matched API columns appended: {', '.join(api_cols)}")
print(f"Repos with API columns retained (instru/ci=True): {int(mask_use_api.sum())} / {len(df_merged)}")


✅ Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\4.1_Total_Repo_Dataset.csv
Matched API columns appended: api_10, api_14, api_15, api_16, api_17, api_18, api_19, api_20, api_21, api_22, api_23, api_24, api_25, api_26, api_27, api_28, api_29, api_30, api_31, api_32, api_33, api_34, api_35, api_36, api_level_total
Repos with API columns retained (instru/ci=True): 459 / 4518
